# 01 · Data Collection — Web Scraping MagicBricks

The data for this project was collected by scraping residential sale listings from [MagicBricks](https://www.magicbricks.com), an Indian property marketplace.

**Tech stack:** Python · Selenium (renders the JavaScript-heavy pages) · BeautifulSoup (parses the HTML) · Pandas (stores the results)

**Output:** `data/raw/MagicBricks_Raw.xlsx` — 2,989 listings × 27 columns across 59 Indian cities.

> The scraper lives in [`src/scraper.py`](../src/scraper.py) so it can be run from the command line. It needs Chrome and live internet access, so this notebook explains it and checks the parser offline instead of re-scraping. Re-running it today may return different listings, and the site's HTML class names may have changed since the data was collected.

## How the scraper works

1. Build the search URL for each `city` and `page`.
2. Load it in a Chrome browser controlled by Selenium, then wait a randomised 2–4 seconds so we don't overload the site.
3. Parse the page with BeautifulSoup and find every listing card (`div.mb-srp__list`).
4. For each card, collect:
   - **Card headers:** title, society name and price.
   - **Hidden `<meta itemprop=…>` tags:** description, URL, locality, latitude and longitude.
   - **Summary label/value pairs:** Super Area, Carpet Area, Status, Floor, Transaction, Furnishing, Bathroom, Car Parking and so on.
5. Save all records to a timestamped Excel file in `data/raw/`.

Run it from the project root:

```bash
python src/scraper.py --cities Pune Mumbai --pages 2
```

In [1]:
import sys
sys.path.append('../src')
from bs4 import BeautifulSoup
from scraper import MagicBricksScraper, BASE_URL

print(BASE_URL.format(page=1, city='Pune'))

https://www.magicbricks.com/property-for-sale/residential-real-estate?proptype=Multistorey-Apartment,Builder-Floor-Apartment,Penthouse,Studio-Apartment,Residential-House,Villa&page=1&cityName=Pune


## Offline test of the parser

To check that the parsing logic works without hitting the live site, we run `parse_card` on a sample listing card that mirrors MagicBricks' HTML structure.

In [2]:
sample_card = '''
<div class="mb-srp__list" id="cardid12345678">
  <meta itemprop="description" content="2 Bath, Semi-Furnished flat in Baner">
  <meta itemprop="url" content="https://www.magicbricks.com/propertyDetails/sample">
  <meta itemprop="addressLocality" content="Baner">
  <meta itemprop="latitude" content="18.5590">
  <meta itemprop="longitude" content="73.7868">
  <h2 class="mb-srp__card--title">2 BHK Flat for Sale in Baner, Pune</h2>
  <a class="mb-srp__card__society--name">Green Valley Residency</a>
  <div class="mb-srp__card__price--amount">&#8377;85 Lac</div>
  <div class="mb-srp__card__summary--label">Super Area</div><div class="mb-srp__card__summary--value">1050 sqft</div>
  <div class="mb-srp__card__summary--label">Status</div><div class="mb-srp__card__summary--value">Ready to Move</div>
  <div class="mb-srp__card__summary--label">Transaction</div><div class="mb-srp__card__summary--value">Resale</div>
  <div class="mb-srp__card__summary--label">Bathroom</div><div class="mb-srp__card__summary--value">2</div>
</div>'''

card = BeautifulSoup(sample_card, 'html.parser').find('div', 'mb-srp__list')
record = MagicBricksScraper.parse_card(card, 'Pune')
record

{'Property Id': 'cardid12345678',
 'Title': '2 BHK Flat for Sale in Baner, Pune',
 'Society Name': 'Green Valley Residency',
 'Price': '₹85 Lac',
 'City': 'Pune',
 'Description': '2 Bath, Semi-Furnished flat in Baner',
 'URL': 'https://www.magicbricks.com/propertyDetails/sample',
 'Address': 'Baner',
 'Latitude': '18.5590',
 'Longitude': '73.7868',
 'Super Area': '1050 sqft',
 'Status': 'Ready to Move',
 'Transaction': 'Resale',
 'Bathroom': '2'}

In [3]:
expected = {'Property Id': 'cardid12345678', 'Price': '₹85 Lac', 'Address': 'Baner',
            'Super Area': '1050 sqft', 'Transaction': 'Resale', 'City': 'Pune'}
assert all(record[k] == v for k, v in expected.items()), 'Parser output does not match'
print('Parser test passed ✔')

Parser test passed ✔


## The collected raw dataset

In [4]:
import pandas as pd
raw = pd.read_excel('../data/raw/MagicBricks_Raw.xlsx')
print(raw.shape)
print('Cities covered:', raw['City'].nunique())
raw.head(3)

(2989, 27)
Cities covered: 59


,Property Id,Title,Society Name,Price,Name,Description,URL,Address,City,Latitude,...,Society,Car Parking,Bathroom,Under Construction,Carpet Area,facing,overlooking,Ownership,Balcony,Dimensions
0,cardid51696257,"4 BHK Flat for Sale Luxury Floors, in DLF City...",Luxury Floors,₹5 Cr,"4 BHK Flat for Sale Luxury Floors, in DLF C...","4 Bath,Semi-Furnished,4 floor Miglani Luxury F...",https://www.magicbricks.com/propertyDetails/4-...,DLF City Phase 1,Gurgaon,77.098466,...,Luxury Floors,1 Covered,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,cardid57200639,"3 BHK Flat for Sale Eldeco Acclaim, in Sohna",Eldeco Acclaim,₹92.8 Lac,"3 BHK Flat for Sale Eldeco Acclaim, in Sohna","3 Bath,Unfurnished Sprawled over 13 acres of v...",https://www.magicbricks.com/propertyDetails/3-...,Sohna,Gurgaon,77.069185,...,Eldeco Acclaim,NaN,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,cardid52514647,"2 BHK Flat for Sale The Leaf, in Sector 85",The Leaf,₹1.18 Cr,"2 BHK Flat for Sale The Leaf, in Sector 85","2 Bath,Semi-Furnished Welcome to nature's lap ...",https://www.magicbricks.com/propertyDetails/2-...,Sector 85,Gurgaon,76.953027,...,The Leaf,NaN,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
